# 🔄 Fair Student v2 — Retraining Notebook
### Optimized Training (Batch Size 128, 16GB VRAM)
This notebook retrains the fairness-aware MobileNetV2 student with a stronger fairness penalty (β=0.4) to reduce the FFPR gap.

**Hardware Targets:** 16GB Dedicated VRAM, 24-core CPU.

1. Setup & Imports
2. Configure v2 hyperparameters (Optimized BS=128)
3. Backup v1 checkpoints & load teacher
4. Create fresh student & data loaders (12 Workers)
5. **Train fair distillation v2** (~2-4 hours with optimization)
6. Plot training history
7. Evaluate on test + cross-datasets
8. Compare v1 vs v2

## ⚙️ Cell 0 — Setup & Imports

In [1]:
import os, sys, time, shutil, warnings
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

# Add project root to path
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import (
    DEVICE, MODELS_DIR, SPLITS_DIR, FIGURES_DIR,
    DEMOGRAPHIC_GROUPS, BATCH_SIZE
)
from src.models.xception import build_teacher
from src.models.mobilenetv2 import build_student
from src.data.dataset import create_dataloaders, DeepfakeDataset
from src.training.train_distill import train_fair_distillation

print('✅ Setup & Imports OK')

✅ Setup & Imports OK


## 🖥️ Cell 1 — GPU Check & v2 Config

In [2]:
print('='*55)
print('  ENVIRONMENT')
print('='*55)
print(f'  PyTorch:      {torch.__version__}')
print(f'  Device:       {DEVICE}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'  GPU:          {gpu.name}')
    print(f'  VRAM:         {gpu.total_memory / 1e9:.1f} GB')
else:
    print('  ⚠️  No GPU found!')

# ── v2 Hyperparameters (AGGRESSIVE 16GB VRAM) ─────────────
V2_ALPHA       = 0.5    # distillation weight
V2_BETA        = 0.4    # fairness weight (KEY CHANGE)
V2_GAMMA       = 0.1    # classification weight
V2_TEMPERATURE = 4.0
V2_EPOCHS      = 50
V2_BATCH_SIZE  = 128    # Lowered from 256 for stable validation
V2_MODEL_NAME  = 'fair_student'
USE_CURRICULUM = False

print() 
print('='*55)
print('  v2 RETRAINING CONFIG (OPTIMIZED)')
print('='*55)
print(f'  Batch size:   {V2_BATCH_SIZE}')
print(f'  Workers:      4')
print(f'  Loss: L = {V2_ALPHA}·Ld + {V2_BETA}·Lf + {V2_GAMMA}·Lc')

# Paths
TEACHER_CHECKPOINT = os.path.join(MODELS_DIR, 'xception_teacher_best.pth')
TRAIN_CSV = os.path.join(SPLITS_DIR, 'train.csv')
VAL_CSV   = os.path.join(SPLITS_DIR, 'val.csv')
TEST_CSV  = os.path.join(SPLITS_DIR, 'test.csv')

torch.backends.cudnn.benchmark = True
print('\ncuDNN benchmark mode: ON')

  ENVIRONMENT
  PyTorch:      2.5.1+cu124
  Device:       cuda
  GPU:          NVIDIA RTX 2000 Ada Generation
  VRAM:         17.2 GB

  v2 RETRAINING CONFIG (OPTIMIZED)
  Batch size:   128
  Workers:      4
  Loss: L = 0.5·Ld + 0.4·Lf + 0.1·Lc

cuDNN benchmark mode: ON


## 💾 Cell 2 — Backup & Load Teacher

In [3]:
print('--- Step 1: Backing up v1 checkpoints ---')
BACKUP_DIR = os.path.join(MODELS_DIR, 'v1_backup')
os.makedirs(BACKUP_DIR, exist_ok=True)
FAIR_FILES = ['fair_student_best_auc.pth', 'fair_student_best_fair.pth', 'fair_student_final.pth']
for fname in FAIR_FILES:
    src = os.path.join(MODELS_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(BACKUP_DIR, fname))
        os.remove(src)
        print(f'  ✅ Backed up & Cleared: {fname}')

print('\n--- Step 2: Loading teacher ---')
teacher = build_teacher(pretrained=False, device=DEVICE)
ckpt = torch.load(TEACHER_CHECKPOINT, map_location=DEVICE, weights_only=False)
teacher.load_state_dict(ckpt['model_state_dict'])
teacher.eval()
print(f'  ✅ Teacher loaded — val AUC: {ckpt.get("val_auc", "N/A")}')

--- Step 1: Backing up v1 checkpoints ---

--- Step 2: Loading teacher ---


d:\M3\DeepFake_Research\.venv\lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


[Teacher] XceptionNet loaded:
  Total params:     21.9M
  Trainable params: 21.9M
  Model size:       83.6 MB
  ✅ Teacher loaded — val AUC: 0.9670840116434661


## 📱 Cell 3 — Create Student & DataLoaders

In [4]:
print('--- Loading Data (Parallel) ---')
loaders = create_dataloaders(TRAIN_CSV, VAL_CSV, batch_size=V2_BATCH_SIZE, num_workers=4)
train_loader = loaders['train']
val_loader   = loaders['val']

assert not isinstance(train_loader, str), 'ERROR: Dataloader desync detected!'

print('\n--- Creating Student ---')
student = build_student(pretrained=True, device=DEVICE)
optimizer = optim.AdamW(student.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

print(f'\n✅ Ready: {len(train_loader)} batches (Size 128)')

--- Loading Data (Parallel) ---
[Dataset] Loaded 146871 samples for train
  REAL: 20970, FAKE: 125901
[Dataset] Loaded 31487 samples for val
  REAL: 4500, FAKE: 26987

--- Creating Student ---
[Student] MobileNetV2 loaded:
  Total params:     2.6M
  Trainable params: 2.6M
  Model size:       9.9 MB

✅ Ready: 1147 batches (Size 128)


## 🚀 Cell 4 — Train Fair Distillation v2

In [5]:
print('=================================================================')
print('  STARTING OPTIMIZED TRAINING (VRAM 16GB)')
print('=================================================================')

results = train_fair_distillation(
    student=student, teacher=teacher,
    train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, scheduler=scheduler,
    num_epochs=V2_EPOCHS, alpha=V2_ALPHA, beta=V2_BETA, gamma=V2_GAMMA,
    temperature=V2_TEMPERATURE, curriculum=USE_CURRICULUM,
    model_name=V2_MODEL_NAME, device=DEVICE
)

print(f'\n✅ TRAINING COMPLETE. Best Val AUC: {results["best_auc"]:.4f}')

  STARTING OPTIMIZED TRAINING (VRAM 16GB)

Fairness-Aware Knowledge Distillation
  Student: fair_student
  Loss: α=0.5·Ld + β=0.4·Lf + γ=0.1·Lc
  Temperature: 4.0, Epochs: 50
  AMP (fp16): ✅ ON



Epoch [1/50] (840.0s) β=0.4000
  Train — Loss: 1.2953 (Ld:2.498 Lf:0.052 Lc:0.257)
  Train — Acc: 0.9136, AUC: 0.9335
  Val   — Acc: 0.8853, AUC: 0.9199, AccGap: 0.1097
    Female-Asian        : 0.9066
    Female-Black        : 0.9228
    Female-Other        : 0.8854
    Female-White        : 0.8990
    Male-Asian          : 0.8513
    Male-Black          : 0.9539
    Male-Other          : 0.8443
    Male-White          : 0.8917
  ★ Best AUC: 0.9199 — saved
  ★ Best Fairness (gap=0.1097) — saved



Epoch [2/50] (441.3s) β=0.4000
  Train — Loss: 0.6379 (Ld:1.219 Lf:0.038 Lc:0.131)
  Train — Acc: 0.9557, AUC: 0.9827
  Val   — Acc: 0.9124, AUC: 0.9400, AccGap: 0.0873
    Female-Asian        : 0.9239
    Female-Black        : 0.9634
    Female-Other        : 0.9183
    Female-White        : 0.9160
    Male-Asian          : 0.9009
    Male-Black          : 0.9704
    Male-Other          : 0.8831
    Male-White          : 0.9158
  ★ Best AUC: 0.9400 — saved
  ★ Best Fairness (gap=0.0873) — saved



Epoch [3/50] (440.8s) β=0.4000
  Train — Loss: 0.4583 (Ld:0.872 Lf:0.032 Lc:0.094)
  Train — Acc: 0.9682, AUC: 0.9906
  Val   — Acc: 0.9084, AUC: 0.9364, AccGap: 0.0886
    Female-Asian        : 0.9214
    Female-Black        : 0.9254
    Female-Other        : 0.8979
    Female-White        : 0.9155
    Male-Asian          : 0.8949
    Male-Black          : 0.9759
    Male-Other          : 0.8873
    Male-White          : 0.9134



Epoch [4/50] (441.0s) β=0.4000
  Train — Loss: 0.3758 (Ld:0.714 Lf:0.028 Lc:0.077)
  Train — Acc: 0.9739, AUC: 0.9934
  Val   — Acc: 0.9158, AUC: 0.9449, AccGap: 0.0782
    Female-Asian        : 0.9168
    Female-Black        : 0.9175
    Female-Other        : 0.9148
    Female-White        : 0.9224
    Male-Asian          : 0.9068
    Male-Black          : 0.9759
    Male-Other          : 0.9127
    Male-White          : 0.8976
  ★ Best AUC: 0.9449 — saved
  ★ Best Fairness (gap=0.0782) — saved



...

Epoch [11/50] (435.4s) β=0.4000
  Train — Loss: 0.2218 (Ld:0.418 Lf:0.021 Lc:0.046)
  Train — Acc: 0.9847, AUC: 0.9971
  Val   — Acc: 0.9110, AUC: 0.9423, AccGap: 0.0775
    Female-Asian        : 0.8932
    Female-Black        : 0.8874
    Female-Other        : 0.9187
    Female-White        : 0.9188
    Male-Asian          : 0.9159
    Male-Black          : 0.9649
    Male-Other          : 0.8963
    Male-White          : 0.9086



Epoch [12/50] (437.4s) β=0.4000
  Train — Loss: 0.1844 (Ld:0.346 Lf:0.019 Lc:0.038)
  Train — Acc: 0.9868, AUC: 0.9981
  Val   — Acc: 0.9135, AUC: 0.9374, AccGap: 0.0742
    Female-Asian        : 0.9027
    Female-Black        : 0.8940
    Female-Other        : 0.9194
    Female-White        : 0.9236
    Male-Asian          : 0.9283
    Male-Black          : 0.9682
    Male-Other          : 0.9009
    Male-White          : 0.8959



Epoch [13/50] (437.7s) β=0.4000
  Train — Loss: 0.1716 (Ld:0.322 Lf:0.018 Lc:0.036)
  Train — Acc: 0.9880, AUC: 0.9982
  Val   — Acc: 0.9159, AUC: 0.9393, AccGap: 0.0761
    Female-Asian        : 0.9267
    Female-Black        : 0.9097
    Female-Other        : 0.9231
    Female-White        : 0.9241
    Male-Asian          : 0.9267
    Male-Black          : 0.9682
    Male-Other          : 0.8921
    Male-White          : 0.9027



Epoch [14/50] (436.1s) β=0.4000
  Train — Loss: 0.1621 (Ld:0.303 Lf:0.018 Lc:0.034)
  Train — Acc: 0.9882, AUC: 0.9984
  Val   — Acc: 0.9181, AUC: 0.9416, AccGap: 0.0679
    Female-Asian        : 0.9274
    Female-Black        : 0.8927
    Female-Other        : 0.9271
    Female-White        : 0.9274
    Male-Asian          : 0.9251
    Male-Black          : 0.9605
    Male-Other          : 0.8992
    Male-White          : 0.9020



Epoch [15/50] (435.2s) β=0.4000
  Train — Loss: 0.1612 (Ld:0.301 Lf:0.018 Lc:0.035)
  Train — Acc: 0.9882, AUC: 0.9983
  Val   — Acc: 0.9179, AUC: 0.9403, AccGap: 0.0690
    Female-Asian        : 0.9309
    Female-Black        : 0.9097
    Female-Other        : 0.9198
    Female-White        : 0.9252
    Male-Asian          : 0.9251
    Male-Black          : 0.9627
    Male-Other          : 0.9093
    Male-White          : 0.8937

Early stopping at epoch 15

Fair distillation complete.
  Best AUC: 0.9507
  Best Fairness Gap: 0.0607

✅ TRAINING COMPLETE. Best Val AUC: 0.9507
